# Representatividade Espacial

 Avaliar a representatividade espacial das estações de monitoramento da qualidade do ar, verificando se as vias próximas a cada estação estão dentro das distâncias adequadas para cada classe de representatividade (micro, meso, bairro e urbana), considerando o poluente e o fluxo de veículos (ADT).

# 1.0 Importando pacotes e funções

Preparo do ambiente de trabalho do notebook, importando todas as bibliotecas necessárias e definindo uma configuração de exibição para números decimais.

In [47]:
rootPath = os.path.dirname((os.getcwd()))
print(rootPath)

/home/nobre/Notebooks/Guia_RQAr/Estadual


In [5]:
# Pacotes e funções
import geopandas as gpd
from pathlib import Path
from long_2_utm_zone import long_2_utm_zone
from utm_zone_2_epsg import utm_zone_2_epsg
import pandas as pd
import numpy as np
import os

# Desativando notação científica
pd.set_option('display.float_format', '{:.2f}'.format)

# 2.0 Definindo caminhos

In [6]:
# Caminhos
root_path = os.path.dirname((os.getcwd()))

# Caminho da pasta de inputs
inputs_path = "https://arquivos.lcqar.ufsc.br/rqar-national-guide-files/data/rep_espacial/inputs"

# Caminho da pasta de outputs
outputs_path = root_path + '/data/rep_espacial/outputs'
os.makedirs(outputs_path, exist_ok=True)

# Arquivo de todas as vias do BR
roads_path = inputs_path + '/processed_roads_dissolved.parquet'

# Arquivo de códigos de vias e os respecitvos fluxo médios diários em um
# buffer de 250 metros das estações
flow_path = inputs_path + '/stations_may_jun_2025.parquet'

# Arquivo de indústrias BR (Gerais, mineração e aterros)
industrial_path = inputs_path + '/industrial_sites_20250902.gpkg'

# Planilha de estações de monitoramento do BR
stations_path = "https://arquivos.lcqar.ufsc.br/rqar-national-guide-files/data/Monitoramento_QAr_BR.csv"

In [7]:
import urllib.request
import tempfile

# Função auxiliar para ler arquivos HTML longos
def read_remote_parquet(url, geo=True):
    with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tmp:
        urllib.request.urlretrieve(url, tmp.name)
        return (gpd.read_parquet(tmp.name) if geo else pd.read_parquet(tmp.name))

# 3.0 Geodataframes de Vias do Brasil

## 3.1 Lendo arquivo do modelo estático de vias

Lê a base geoespacial da malha viária do Brasil.

Arquivo de entrada:
- processed_roads_dissolved.parquet

Colunas obrigatórias:
- osm_id: identificador único da via no OpenStreetMap.
- geometry: geometria da via (LineString ou MultiLineString).

Processamentos realizados:
- Carrega o arquivo Parquet como um GeoDataFrame.
- Transforma o índice em uma coluna do GeoDataFrame.
- Converte a coluna 'osm_id' para o tipo inteiro.

Saída:
 - GeoDataFrame 'roads', contendo as informações espaciais das vias que serão utilizadas nas etapas posteriores do processamento.

In [8]:
roads = read_remote_parquet(roads_path).reset_index(drop=False).astype({"osm_id": int})
roads.head()

,osm_id,geometry,name,highway,lanes,oneway,surface,maxspeed
0,4217292,"MULTILINESTRING ((-43.20285 -22.98436, -43.202...",Rua Vinícius de Moraes,residential,2,yes,asphalt,<NA>
1,4217293,"MULTILINESTRING ((-43.20511 -22.98637, -43.205...",Rua Joana Angélica,residential,2,yes,asphalt,<NA>
2,4217297,"MULTILINESTRING ((-43.20679 -22.98073, -43.206...",Rua Maria Quitéria,residential,2,yes,asphalt,<NA>
3,4217299,"MULTILINESTRING ((-43.20686 -22.98095, -43.206...",Rua Alberto de Campos,residential,2,yes,asphalt,<NA>
4,4217303,"MULTILINESTRING ((-43.21387 -22.98212, -43.213...",Rua Redentor,residential,1,yes,asphalt,<NA>


## 3.2 Lendo os dados de fluxo de veículos e cálculo de ADT

Lê a base de dados de fluxo de veículos das vias localizadas em um raio de 250 m das estações de monitoramento.

O ADT será calculado nas etapas seguintes:
- ADT significa Average Daily Traffic, ou Tráfego Médio Diário (TMD).

In [9]:
roads_with_adt = pd.read_parquet(flow_path)
roads_with_adt.head()

,osm_id,weekday,hour,traffic_level,vehicle_count,vehicle_count_max,vehicle_count_min,highway
0,143542915.00,0,0,40.70,0.00,0.00,0.00,tertiary
1,143542915.00,0,1,40.70,0.00,0.00,0.00,tertiary
2,143542915.00,0,2,40.70,0.00,0.00,0.00,tertiary
3,143542915.00,0,3,40.70,0.00,0.00,0.00,tertiary
4,143542915.00,0,4,40.70,0.00,0.00,0.00,tertiary


In [10]:
# Define os nomes das colunas utilizadas no cálculo do ADT
adt_col = 'average_daily_vehicle_count'
vehicle_count_col = 'vehicle_count'

# Soma o número de veículos por via e por dia da semana e calcula a média por via.
roads_with_adt = roads_with_adt.groupby(['osm_id','weekday'])['vehicle_count'].sum()
roads_with_adt = roads_with_adt.groupby('osm_id').mean().reset_index()

# Renomeia a coluna para identificar o valor como ADT
roads_with_adt = roads_with_adt.rename({vehicle_count_col : adt_col}, axis=1)
roads_with_adt.head()

,osm_id,average_daily_vehicle_count
0,4217292.00,20775.76
1,4217293.00,18657.09
2,4217297.00,18560.23
3,4217299.00,20331.30
4,4217303.00,3830.52


## 3.3 Selecionando vias com ADT calculado e > 1000 veículos

In [11]:
# Junta o ADT calculado à base de vias usando o identificador OSM da via. Apenas vias presentes nas duas bases são mantidas.

roads = pd.merge(roads, roads_with_adt, how='inner', on='osm_id')
roads.head()

,osm_id,geometry,name,highway,lanes,oneway,surface,maxspeed,average_daily_vehicle_count
0,4217292,"MULTILINESTRING ((-43.20285 -22.98436, -43.202...",Rua Vinícius de Moraes,residential,2,yes,asphalt,<NA>,20775.76
1,4217293,"MULTILINESTRING ((-43.20511 -22.98637, -43.205...",Rua Joana Angélica,residential,2,yes,asphalt,<NA>,18657.09
2,4217297,"MULTILINESTRING ((-43.20679 -22.98073, -43.206...",Rua Maria Quitéria,residential,2,yes,asphalt,<NA>,18560.23
3,4217299,"MULTILINESTRING ((-43.20686 -22.98095, -43.206...",Rua Alberto de Campos,residential,2,yes,asphalt,<NA>,20331.30
4,4217303,"MULTILINESTRING ((-43.21387 -22.98212, -43.213...",Rua Redentor,residential,1,yes,asphalt,<NA>,3830.52


*Definiu-se 1000 veículos/dia como o fluxo diário médio (ADT) mínimo para uma via ser considerada como via principal, termo utilizado no Guia de Monitoramento da Qualidade do Ar do Brasil.*

In [12]:
# Pegando vias com ADT superior a 1000 veículos/dia

roads = roads.loc[roads[adt_col] > 1000, :]
roads.head()

,osm_id,geometry,name,highway,lanes,oneway,surface,maxspeed,average_daily_vehicle_count
0,4217292,"MULTILINESTRING ((-43.20285 -22.98436, -43.202...",Rua Vinícius de Moraes,residential,2,yes,asphalt,<NA>,20775.76
1,4217293,"MULTILINESTRING ((-43.20511 -22.98637, -43.205...",Rua Joana Angélica,residential,2,yes,asphalt,<NA>,18657.09
2,4217297,"MULTILINESTRING ((-43.20679 -22.98073, -43.206...",Rua Maria Quitéria,residential,2,yes,asphalt,<NA>,18560.23
3,4217299,"MULTILINESTRING ((-43.20686 -22.98095, -43.206...",Rua Alberto de Campos,residential,2,yes,asphalt,<NA>,20331.30
4,4217303,"MULTILINESTRING ((-43.21387 -22.98212, -43.213...",Rua Redentor,residential,1,yes,asphalt,<NA>,3830.52


## 3.4 Verificando se alguma das vias não foi contemplada com dados de fluxo

In [13]:
if roads[adt_col].isna().any():
    print('Ops! Verificar!')
else:
    print('Pode seguir tranquile!')

Pode seguir tranquile!


# 4.0 Zonas Industriais

--> Informação complementar (Indústria mais próxima a estação)

## 4.1 Lendo o arquivo de indústrias

Nesta etapa, é carregada a base geoespacial contendo indústrias, áreas de mineração e aterros sanitários.

A base deve conter, obrigatoriamente, nome comercial do empreendimento com coluna **'Razão Social': str** e geometria com coluna **'geometry': Point ou Polygon**.

As geometrias representam a localização ou área dos empreendimentos e serão *utilizadas posteriormente nas análises espaciais*.

In [14]:
# Lendo arquivo de indústrias
industrial_gdf = gpd.read_file(industrial_path)

# Duplicando a coluna de geometria para transmitir ela após o sjoin_nearest na seção 6.2
industrial_gdf['industry_geom'] = industrial_gdf.geometry
industrial_gdf.head()

,CNPJ,Razão Social,Código da categoria,Descrição da categoria,Código da atividade,Descrição da atividade,Data de início da atividade,Data de término da atividade,Potencial de Poluição da atividade,Município,UF,Latitude,Longitude,Situação cadastral,silt_loading,geometry,industry_geom
0,8673000198,JAGUAR EQUIPAMENTOS ELETRO INDUSTRIAL LTDA - EPP,3.00,Indústria Metalúrgica,1.00,Fabricação de aço e de produtos siderúrgicos,01/01/2001,02/12/2003,Alto,PINHALZINHO,SAO PAULO,-22.782972,-46.570606,Ativa,9.70,POINT (-46.57061 -22.78297),POINT (-46.57061 -22.78297)
1,98809000106,ENGEFLAT COMÉRCIO E ENGENHARIA LTDA.,14.00,Indústrias Diversas,1.00,Usinas de produção de concreto,17/06/1994,28/09/2022,Pequeno,BELO HORIZONTE,MINAS GERAIS,-19.920806,-43.937778,Ativa,120.00,POINT (-43.93778 -19.92081),POINT (-43.93778 -19.92081)
2,148025000218,LATASA INDUSTRIA E COMERCIO LTDA,14.00,Indústrias Diversas,1.00,Usinas de produção de concreto,01/01/2002,13/03/2002,Pequeno,PINDAMONHANGABA,SAO PAULO,-22.923889,-45.461667,Ativa,120.00,POINT (-45.46167 -22.92389),POINT (-45.46167 -22.92389)
3,148025000218,LATASA INDUSTRIA E COMERCIO LTDA,14.00,Indústrias Diversas,2.00,Usinas de produção de asfalto,01/01/2002,13/03/2002,Pequeno,PINDAMONHANGABA,SAO PAULO,-22.923889,-45.461667,Ativa,12.00,POINT (-45.46167 -22.92389),POINT (-45.46167 -22.92389)
4,177693000192,SAINT LUIGER PROCESSADORA DE ALIMENTOS LTDA,16.00,Indústria de Produtos Alimentares e Bebidas,1.00,"Beneficiamento, moagem, torrefação e fabricaçã...",22/08/1994,23/01/2017,Médio,COTIA,SAO PAULO,-23.597381,-46.886322,Ativa,1.10,POINT (-46.88632 -23.59738),POINT (-46.88632 -23.59738)


# 5.0 Estações de monitoramento da qualidade do ar

## 5.1 Lendo arquivo, determinando código EPSG e filtragem de poluentes de interesse

Esta seção faz a leitura do arquivo de estações de monitoramento da qualidade do ar 
do Brasil, com as colunas obrigatórias:
- 'LONGITUDE': float.
- 'LATITUDE': float.
- 'COD_POLUENTE': float
- 'POLUENTE': str.
- 'ID_OEMA': str.

Em seguida, cada estação é enquadrada dentro de uma zona UTM e atribui-se o código EPSG correspondente, de acordo com a zona UTM e a latitude de cada uma.

Por fim, dentre todas as linhas de estações, o geodataframe é reduzido às que monitoram os poluentes a seguir:
- monóxido de carbono (CO)
- dióxido de enxofre (SO2)
- dióxido de nitrogênio (NO2)
- ozônio (O3)
- material particulado de diâmetro inferior a 10 micrômetros (MP10)
- material particulado de diâmetro inferior a 2.5 micrômetros (MP2.5)
- material particulado total (PTS)

In [15]:
# Lendo o arquivo de estações de monitoramento
stations = pd.read_csv(filepath_or_buffer=stations_path,
                      dtype={'LONGITUDE': float,
                             'LATITUDE':float,
                             'COD_POLUENTE':float,
                             'POLUENTE':str,
                             'ID_OEMA':str
                            }
                      )

# Transformando em GeoDataFrame
stations = gpd.GeoDataFrame(stations,
                            geometry=gpd.points_from_xy(stations.LONGITUDE,
                                                        stations.LATITUDE,
                                                        crs='EPSG:4326'))
# Determinando a zona UTM para cada estação
stations.loc[:,'utm_zone'] = long_2_utm_zone(stations
                                             .geometry
                                             .centroid
                                             .x)

# Determinando do código EPSG para cada estação
stations.loc[:,'EPSG'] = utm_zone_2_epsg(stations['utm_zone'],
                                         stations.geometry
                                         .centroid
                                         .x)

# Removendo a coluna auxiliar de zona UTM
stations.drop(columns='utm_zone', inplace=True)

# Filtrando as estações que monitoram CO, SO2, O3, NO2, PM10, PM2.5 e PTS
stations = stations[stations['COD_POLUENTE'].isin([1.0, 2.0, 3.0, 4.0,
                                                   5.0, 7.0, 8.0])]
stations.head()

/tmp/ipykernel_585375/1591723292.py:19: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .centroid
/tmp/ipykernel_585375/1591723292.py:25: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .centroid


,UF,CIDADE,CD_MUN,ID_OEMA,ID_MMA,ID_MMA_COMPLETO,PROPRIETARIO,PROP_ENTIDADE,OPERADOR,OP_ENTIDADE,...,ELEVACAO,REALOCACAO,OBS_CALIBRACAO,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,REP_ESPACIAL_DECLARADA,OPERACAO,geometry,EPSG
0,AC,Porto Acre,1200807,MPAC_PTA_01_Sec.infraestrutura,AC0017,AC0017RA002,Nao declarado,Publico,Nao declarado,Nao declarado,...,206.00,Nao declarado,Nao declarado,Nao declarado,Sim,Nao declarado,Nao declarado,Nao declarado,POINT (-67.69802 -9.72708),EPSG:31979
1,AC,Feijó,1200302,MPAC_FIJ_01_promotoria,AC0012,AC0012RA002,Nao declarado,Publico,Nao declarado,Nao declarado,...,160.00,Nao declarado,Nao declarado,Nao declarado,Sim,Nao declarado,Nao declarado,Nao declarado,POINT (-70.35503 -8.17008),EPSG:31979
2,AC,Bujari,1200138,MPAC_BJR_01_promotoria,AC0006,AC0006RA002,Nao declarado,Publico,Nao declarado,Nao declarado,...,212.00,Nao declarado,Nao declarado,Nao declarado,Sim,Nao declarado,Nao declarado,Nao declarado,POINT (-67.95388 -9.82918),EPSG:31979
3,AC,Brasiléia,1200104,MPAC_BRL_01_promotoria,AC0007,AC0007RA002,Nao declarado,Publico,Nao declarado,Nao declarado,...,194.00,Nao declarado,Nao declarado,Nao declarado,Sim,Nao declarado,Nao declarado,Nao declarado,POINT (-68.74228 -11.01051),EPSG:31979
4,AC,Tarauacá,1200609,MPAC_TRC_02_ifac,AC0024,AC0024RA002,Nao declarado,Publico,Nao declarado,Nao declarado,...,189.00,Nao declarado,Nao declarado,Nao declarado,Sim,Nao declarado,Nao declarado,Nao declarado,POINT (-70.7709 -8.14315),EPSG:31979


# 6.0 Distância de vias de vários ADTs e indústrias às estações

## 6.1 Pegando via mais próxima de cada estação

In [16]:
# # Criando dicionários de subsets para cada EPSG 
# roads_subsets = {}
# stations_subsets = {}
# industries_subsets = {}

# # Duplicando coluna de geometria das ruas para preservá-la no sjoin
# roads['road_geom'] = roads.geometry

# # Criando dicionário de epsg
# choices = {'{}'.format(q): q for q in stations['EPSG'].unique()}

# # Iterando por códigos EPSG
# for epsg in choices.keys():
    
#      # Criando sub dataframes de estações e definindo o SRC
#      stations_subsets[epsg] = (
#          stations[stations['EPSG'] == epsg]
#          .to_crs(epsg)
#      )

#      # Criando sub dataframes de vias para cada EPSG
#      roads_subsets[epsg] = roads[['osm_id','road_geom','geometry']].to_crs(epsg)

#      # Calculando distâncias da via mais próxima, para cada faixa de ADT
#      stations_subsets[epsg] = (
#         gpd.sjoin_nearest(stations_subsets[epsg],
#                           roads_subsets[epsg],
#                           how='left',
#                           distance_col='distance_m')
#      )

#      # Reprojetando geometria de cada subdataframe para WGC 84 
#      stations_subsets[epsg] = stations_subsets[epsg].to_crs(4326)

# distances = (
#     gpd.GeoDataFrame(pd.concat([stations_subsets[df] 
#                                 for df 
#                                 in stations_subsets])
#                          )
#         )

# # Removendo linhas duplicadas de uma mesma estação
# distances.drop_duplicates(subset='ID_OEMA', inplace=True)

# # Salvando para csv de distâncias de ruas a estações
# distances.to_parquet(outputs_path+'/distances_to_roads.parquet')

## 6.1 Funções


Esta seção faz a junção espacial entre estações e as vias mais próximas para a cada, dentro de cada faixa de ADT, segundo as faixas disponíveis para cada poluente. Então, 3 colunas são criadas para cada faixa de ADT com vias de ADT > 0: 
- 'osm_id_{}k': int.
    - Código de indentificação de vias do OPenStreetMaps
- 'average_daily_vehicle_count_{}k': float.
    - Fluxo médio diário de veículos para a via mais próxima dentro de cada faixa de ADT
- 'distance_{}k': float.
    - Distância de cada estação para cada via, em metros


In [17]:
def get_distance_to_stations(stations:gpd.GeoDataFrame,
                             poll:str,
                             roads:gpd.GeoDataFrame,
                             industries:gpd.GeoDataFrame,
                             get_industries:bool=True):

    """
    
    Parameters
    ----------
    stations : gpd.GeoDataFrame
        GeoDataFrame contendo as estações de monitoramento da qualidade do ar.
    
    poll : str
        Poluente utilizado para selecionar as faixas de ADT correspondentes.
        Valores possíveis: 'co', 'so2', 'no2', 'pm' e 'o3'.
    
    roads : gpd.GeoDataFrame
        GeoDataFrame contendo as vias, com as colunas 'average_daily_vehicle_count'
        e 'osm_id'.
    
    industries : gpd.GeoDataFrame
        GeoDataFrame contendo as indústrias ativas e suas geometrias.
    
    get_industries : bool, optional
        Se True, também calcula a distância até a indústria mais próxima de cada
        estação e adiciona as colunas 'distance_to_industry', 'industry_geom'
        e 'Razão Social'. O padrão é False.
    
    Returns
    -------
    gpd.GeoDataFrame
        GeoDataFrame de entrada com três colunas adicionais para cada faixa de
        ADT que possua vias disponíveis: 'osm_id_{}k',
        'average_daily_vehicle_count_{}k' e 'distance_{}k'.
    
        Quando get_industries=True, também são adicionadas as colunas
        'distance_to_industry', 'industry_geom' e 'Razão Social', referentes
        à indústria mais próxima de cada estação.
    """

    # Dicionário de faixas de ADT (value * 1000)
    pollutants_adt_dict = {'co':[1, 10, 20, 30, 40, 50, 60, np.inf],
                           'so2':[1, 10, 20, 30, 40, 50, 60, np.inf],
                           'no2':[1, 10, 15, 20, 40, 70, 110, np.inf],
                           'pm': [1, 15, 20, 30, 40, 50, 60, 70, 80, np.inf],
                           'o3':[10, 15, 20, 40, 70, 110, np.inf]}

    # Dicionário de subsets por poluente
    poll_codes = {
        'co': [7],
        'so2': [3],
        'no2': [4],
        'pm': [1,2,8],
        'o3': [5]
    }

    # Filtrando subsets de estações e adt_list para o poluente
    poll_subset = stations[stations['COD_POLUENTE'].isin(poll_codes[poll])]
    adt_list = pollutants_adt_dict[poll]

    # Removendo estações com geometria inválida
    poll_subset = poll_subset[
    poll_subset.geometry.notna() &
    poll_subset.geometry.is_valid]

    adt_list = pollutants_adt_dict[poll]

    # Criando dicionários de subsets para cada EPSG 
    roads_subsets = {}
    stations_subsets = {}
    industries_subsets = {}
    
    # Criando dicionário de EPSG apenas para as estações válidas
    choices = {'{}'.format(q): q for q in poll_subset['EPSG'].unique()}

    # Iterando por códigos EPSG
    for epsg in choices.keys():
        
         # Criando sub dataframes de estações e definindo o SRC
         stations_subsets[epsg] = (
             poll_subset[poll_subset['EPSG'] == epsg]
             .to_crs(epsg)
         )
    
         # Criando sub dataframes de vias para cada EPSG
         roads_subsets[epsg] = roads.to_crs(epsg)
    
         # Criando sub dataframe de indústrias para cada EPSG
         industries_subsets[epsg] = industries.to_crs(epsg)
         
         # Iterando sobre valores de ADT
         for idx, adt in enumerate(adt_list[0:-1]):
             filtered_roads = (
                 roads_subsets[epsg]
                 .loc[(roads_subsets[epsg][adt_col] >= adt * 1000) &
                      (roads_subsets[epsg][adt_col] < adt_list[idx + 1] * 1000),
                 ['osm_id', adt_col, 'geometry']]
                 )
             
             if filtered_roads.empty:
                 continue

             # Calculando distâncias da via mais próxima, para cada faixa de ADT
             stations_subsets[epsg] = (
                gpd.sjoin_nearest(stations_subsets[epsg],
                                  filtered_roads,
                                  how='left',
                                  lsuffix=('{}k'.format(adt_list[idx - 1])
                                                        if idx > 0
                                                        else None),
                                  rsuffix='{}k'.format(adt),
                                  distance_col='distance_{}k'.format(adt))
                 )
     
             # Removendo right index do sjoin
             stations_subsets[epsg].drop(columns='index_{}k'.format(adt),
                                         inplace=True)
             
             rename_cols = {}
             if 'osm_id' in stations_subsets[epsg].columns:
                 rename_cols['osm_id'] = f'osm_id_{adt}k'
             if adt_col in stations_subsets[epsg].columns:
                 rename_cols[adt_col] = f'{adt_col}_{adt}k'
             if rename_cols:
                 stations_subsets[epsg].rename(columns=rename_cols, inplace=True)
             
             # Removendo linhas duplicadas pelo fato de existir diversas vias
             # com a mesma distância da estação
             if f'{adt_col}_{adt}k' in stations_subsets[epsg].columns:
                 stations_subsets[epsg] = (
                     stations_subsets[epsg]
                     .sort_values(by=f'{adt_col}_{adt}k',ascending=False)
                     .drop_duplicates(subset=['ID_OEMA',
                                              f'distance_{adt}k',
                                              'POLUENTE'])
                  )
            
                        
         if get_industries == True:
             # Calculando a distância da estação para a indústria mais próxima
             stations_subsets[epsg] = (
                 gpd.sjoin_nearest(stations_subsets[epsg],
                                   industries_subsets[epsg][['Razão Social',
                                                             'industry_geom',
                                                             'geometry']],
                                   distance_col="distance_to_industry")
                 )
         
         # Removendo duplicatas com todas as colunas duplicadas
         stations_subsets[epsg] = (
             stations_subsets[epsg]
             .drop_duplicates(subset= ['ID_OEMA', 'POLUENTE']))
         
         # Removendo right index do sjoin 
         stations_subsets[epsg].drop(columns='index_right', inplace=True)
         
         # Reprojetando geometria de cada subdataframe para WGC 84 
         stations_subsets[epsg] = stations_subsets[epsg].to_crs(4326)
         
    # Concatenando sub geodataframes
    stations_by_poll = (
        gpd.GeoDataFrame(pd.concat([stations_subsets[df] 
                                    for df 
                                    in stations_subsets])
                         )
        )
    
    return stations_by_poll

## 6.2 Aplicando função

A função é aplicada para cada poluente de interesse, gerando um GeoDataFrame específico para cada grupo de poluentes.

Em cada resultado são calculadas as distâncias entre as estações e as vias mais próximas em diferentes faixas de ADT, além da distância até a indústria mais próxima.

In [18]:
# Aplicando função
subset_co = get_distance_to_stations(stations=stations,
                                     poll='co',
                                     roads=roads,
                                     industries=industrial_gdf,
                                     get_industries=True)

subset_no2 = get_distance_to_stations(stations=stations,
                                      poll='no2',
                                      roads=roads,
                                      industries=industrial_gdf,
                                      get_industries=True)

subset_o3 = get_distance_to_stations(stations=stations,
                                     poll='o3',
                                     roads=roads,
                                     industries=industrial_gdf,
                                     get_industries=True)

subset_pm = get_distance_to_stations(stations=stations,
                                     poll='pm',
                                     roads=roads,
                                     industries=industrial_gdf,
                                     get_industries=True)

subset_so2 = get_distance_to_stations(stations=stations,
                                      poll='so2',
                                      roads=roads,
                                      industries=industrial_gdf,
                                      get_industries=True)

In [19]:
# Exemplo de verificação das distâncias calculadas para uma estação específica

subset_pm.loc[subset_pm.ID_OEMA.str.contains('Fercal CRAS')].iloc[[1]][[col for col in list(subset_pm.columns) if 'distance' in col]]
#subset_o3.loc[subset_o3.ID_MMA.str.contains('SC0004')]

,distance_1k,distance_15k,distance_20k,distance_30k,distance_40k,distance_50k,distance_60k,distance_70k,distance_80k,distance_to_industry
324,155.88,965.86,3715.17,4860.47,6152.88,12715.47,8105.77,8904.17,2010.87,8039.90


# 7.0 Classificação de representatividade espacial

Para a classificação, existem diferentes tabelas de referência para cada poluente, mostrando as distâncias mínimas e máximas que a estação deve estar das vias principais mais próximas, de acordo com os fluxos médios diários das vias (ADT). Essas tabelas impõem distâncias para os valores de ADT de 15k, 20k, 40, etc, e todos os valores intermediários devem ser interpolados.

Esta seção visa à organização das tabelas de referência, à adição de linhas de valores de ADT intermediários e à interpolação das distâncias mínimas e máximas para cada um deles. Depois disso, os limites de distância para cada fluxo serão adicionados a cada dataframe de poluente e as distâncias de cada via para cada estação serão classificadas como True se se encontrarem dentro desses limites. Isso cria colunas de True/False para cada classe de representatividade espacial existente na tabela de referência de cada poluente (micro, meso, bairro, urbana).

Como passo final, cada linha (via) será enquadrada dentro de uma das 4 classes de representatividade espacial, seguindo a ordem de prioridade micro > meso > bairro > urbana, logo a classe escolhida será sempre a classe mais restritiva. Por exemplo, se a via tem valores True tanto para micro, bairro e urbana, ela será classificada como micro.

--> **Para essa etapa, as funções e linhas de códigos são mais detalhadas para que o usuário possa entender o passo a passo**

## 7.1 Montando tabelas de referência

### --> Transformar em links externos os arquivos abaixo:

In [20]:
# Lendo os arquivos para cada tabela
# https://www.gov.br/mma/pt-br/assuntos/meio-ambiente-urbano-recursos-hidricos-qualidade-ambiental/qualidade-do-ar/guia-tecnico-para-o-monitoramento-e-avaliacao-da-qualidade-do-ar.pdf

ref_table_co = pd.read_csv(inputs_path + '/ref_table_so2eco.csv')
ref_table_no2 = pd.read_csv(inputs_path + '/ref_table_no2.csv')
ref_table_o3 = pd.read_csv(inputs_path + '/ref_table_o3.csv')
ref_table_pm = pd.read_csv(inputs_path + '/ref_table_pm.csv')
ref_table_so2 = pd.read_csv(inputs_path + '/ref_table_so2eco.csv')

# Nota da EPA acerca da ocorrência de valores intermediários de ADT
'''
Distance from the edge of the nearest traffic lane. The distance for 
intermediate traffic counts should be interpolated from the table values based
on the actual traffic count.

Distância da borda da via mais próxima. A distância para contagens de veículos (ADT)
intermediárias deve ser interpolada a partir dos valores das tabelas baseados na
contagem de veículos observada.

# https://www.ecfr.gov/current/title-40/chapter-I/subchapter-C/part-58/appendix-Appendix%20E%20to%20Part%2058
'''

# Dicionário de faixas de ADT (real = valor * 1000)
pollutants_adt_dict = {'co':[1, 10, 20, 30, 40, 50, 60, np.inf],
                       'so2':[1, 10, 20, 30, 40, 50, 60, np.inf],
                       'no2':[1, 10, 15, 20, 40, 70, 110, np.inf],
                       'pm': [1, 15, 20, 30, 40, 50, 60, 70, 80, np.inf],
                       'o3':[10, 15, 20, 40, 70, 110, np.inf]}

# Definindo coluna de adt como índice
interpolated_co = ref_table_co.set_index('avg_adt').squeeze()
interpolated_no2 = ref_table_no2.set_index('avg_adt').squeeze()
interpolated_o3 = ref_table_o3.set_index('avg_adt').squeeze()
interpolated_pm = ref_table_pm.set_index('avg_adt').squeeze()
interpolated_so2 = ref_table_so2.set_index('avg_adt').squeeze()

In [21]:
interpolated_no2

,micro_min,micro_max,bairro_min,bairro_max,urb_min,urb_max
avg_adt,,,,,,
1000,2,10,10,inf,10,inf
10000,2,10,10,inf,10,inf
15000,2,10,20,inf,20,inf
20000,2,10,30,inf,30,inf
40000,2,10,50,inf,50,inf
70000,2,10,100,inf,100,inf
110000,2,10,250,inf,250,inf


## 7.2 Interpolação dos limites de distância para cada classe de representatividade espacial

In [22]:
# # Montando dicionário de dataframes para interpolação
# interpolated_dict = {
#     'co': interpolated_co,
#     'so2': interpolated_so2,
#     'no2': interpolated_no2,
#     'pm': interpolated_pm,
#     'o3': interpolated_o3
# }

# # Redefinindo dicionário de subsets de poluentes
# pollutant_subsets = {
#     'co': subset_co,
#     'so2': subset_so2,
#     'no2': subset_no2,
#     'pm': subset_pm,
#     'o3': subset_o3
# }

# ## Interpolando os limites das classes de representatividade ------------------------
# # Iterando sobre os poluentes e os subsets de poluentes
# for poll, subset in pollutant_subsets.items():
    
#     # Iterando sobre os valores de adt da tabela de referencia de cada poluente
#     for idx, adt_band in enumerate(pollutants_adt_dict[poll][:-1]):
#         col_name = f'average_daily_vehicle_count_{adt_band}k'
        
#         if col_name not in subset.columns:
#             print(f"[WARNING] '{col_name}' nonexistant in subset_{poll}")
#             continue
        
#         # Iterando sobre os valores de adt para cada via do subset do poluente
#         for adt_value in subset[col_name]:
#             interpolated_dict[poll].loc[adt_value] = np.nan
                
#     # Organizar pelo índice de modo ascendente
#     interpolated_dict[poll].sort_index(inplace=True)
                
#     # Interpolando os valores NaN #FIXME
#     cols = [col for col in interpolated_dict[poll].columns if (('min' in col) | ('max' in col))]
#     for col in cols:
#         vals = set(interpolated_dict[poll][col].dropna().unique())
#         if len(vals) == 1 and interpolated_dict[poll][col].isna().any():
#             interpolated_dict[poll][col] = interpolated_dict[poll][col].dropna().unique()[0]
#         else:
#             interpolated_dict[poll][col] = pd.to_numeric(interpolated_dict[poll][col], errors='coerce')
#             interpolated_dict[poll][col] = interpolated_dict[poll][col].interpolate(method='index',
#                                                                                     limit_area='inside')
#             interpolated_dict[poll][col] = interpolated_dict[poll][col].interpolate(method='spline',
#                                                                                     order=1,
#                                                                                     limit_direction='forward')
#             interpolated_dict[poll][col] = interpolated_dict[poll][col].fillna(np.inf)

    
#     # Resetando index
#     interpolated_dict[poll].reset_index(inplace=True)

In [23]:
# interpolated_dict['pm']

São calculados os limites de distância das classes de representatividade (micro, meso, bairro e urbano) para os diferentes valores de ADT (fluxo médio diário de veículos).

A função `interp_limites_rep()` realiza esse procedimento separadamente para cada poluente, utilizando as faixas de ADT definidas no dicionário `pollutants_adt_dict` e os valores de distância disponíveis na tabela de referência.

O procedimento consiste em:

1. Definir as faixas de ADT consideradas para cada poluente;
2. Identificar, no `subset`, a coluna correspondente ao ADT de cada faixa;
3. Associar os valores de ADT das vias aos respectivos limites de distância da tabela de referência;
4. Organizar os valores de ADT em ordem crescente;
5. Identificar os valores ausentes (`NaN`) nos limites de distância;
6. Realizar uma interpolação linear (`index`) e, posteriormente, uma interpolação por *spline* para estimar limites nos valores de ADT que não possuem um valor diretamente disponível na tabela;
7. Preencher os valores que permanecem ausentes com `np.inf`, indicando a ausência de um limite superior definido;
8. Restaurar o índice do `DataFrame`.

Dessa forma, a função gera uma tabela com os limites de distância estimados para diferentes valores de ADT, permitindo posteriormente comparar a distância entre cada estação e a via com os limites correspondentes às classes de representatividade.

In [24]:
## Interpolando os limites das classes de representatividade ------------------------
def interp_limites_rep(subset: gpd.GeoDataFrame,
                       interpolated: pd.DataFrame,
                       poluente: str) -> pd.DataFrame:
  
    # Dicionário de faixas de ADT (value * 1000)
    pollutants_adt_dict = {'co':[1, 10, 20, 30, 40, 50, 60, np.inf],
                           'so2':[1, 10, 20, 30, 40, 50, 60, np.inf],
                           'no2':[1, 10, 15, 20, 40, 70, 110, np.inf],
                           'pm': [1, 15, 20, 30, 40, 50, 60, 70, 80, np.inf],
                           'o3':[10, 15, 20, 40, 70, 110, np.inf]}
    
    # Iterando sobre os valores de adt da tabela de referencia de cada poluente
    for idx, adt_band in enumerate(pollutants_adt_dict[poluente][:-1]):
        col_name = f'average_daily_vehicle_count_{adt_band}k'
        
        if col_name not in subset.columns:
            print(f"[WARNING] '{col_name}' nonexistant in subset_{poluente}")
            continue
        
        # Iterando sobre os valores de adt para cada via do subset do poluente
        for adt_value in subset[col_name]:
            interpolated.loc[adt_value] = np.nan
                
    # Organizar pelo índice de modo ascendente
    interpolated.sort_index(inplace=True)
                
    # Interpolando os valores NaN #FIXME
    cols = [col for col in interpolated.columns if (('min' in col) | ('max' in col))]
    for col in cols:
        vals = set(interpolated[col].dropna().unique())
        if len(vals) == 1 and interpolated[col].isna().any():
            interpolated[col] = interpolated[col].dropna().unique()[0]
        else:
            interpolated[col] = pd.to_numeric(interpolated[col],
                                              errors='coerce')
            
            interpolated[col] = interpolated[col].interpolate(method='index',
                                                              limit_area='inside')
            interpolated[col] = interpolated[col].interpolate(method='spline',
                                                              order=1,
                                                              limit_direction='forward')
            interpolated[col] = interpolated[col].fillna(np.inf)

                
    # Resetando index
    interpolated.reset_index(inplace=True)

    return interpolated

In [25]:
# Aplicando função
interpolated_co = interp_limites_rep(subset_co,
                                     interpolated_co,
                                     'co')
interpolated_no2 = interp_limites_rep(subset_no2,
                                     interpolated_no2,
                                     'no2')
interpolated_o3 = interp_limites_rep(subset_o3,
                                     interpolated_o3,
                                     'o3')
interpolated_pm = interp_limites_rep(subset_pm,
                                     interpolated_pm,
                                     'pm')
interpolated_so2 = interp_limites_rep(subset_so2,
                                     interpolated_so2,
                                     'so2')

## 7.3 Adicionando ao gdf de estradas os limites das tabelas de referência interpoladas

In [26]:
# # Iterando sobre os poluentes e seus subconjuntos
# for poll, subset in pollutant_subsets.items():
    
#     # Iterando sobre os valores de ADT da ref_table para cada poluente
#     for idx, adt_band in enumerate(pollutants_adt_dict[poll][:-1]):
#         col_name = f'average_daily_vehicle_count_{adt_band}k'
        
#         # Registrando valores de ADT sem vias para cada poluente
#         if col_name not in subset.columns:
#             print(f"[AVISO] '{col_name}' não existe no subset_{poll}")
#             continue
        
#         # Obtendo a tabela do lado direito (para o merge)
#         interp_df = interpolated_dict[poll].copy()
        
#         # Renomeando a primeira coluna, para que a função merge não adicione sufixo
#         suffix = f"_{adt_band}k"
#         interp_df = interp_df.rename(columns={
#             col: f"{col}{suffix}" 
#             for col
#             in interp_df.columns
#             if col != 'avg_adt'
#         })
        
#         # Mesclando colunas com limites de distância para cada ADT e classe representativa
#         pollutant_subsets[poll] = pollutant_subsets[poll].merge(
#             right= interp_df,
#             how='left',
#             left_on= col_name,
#             right_on='avg_adt',
#             suffixes=(None, f"_{adt_band}k")
#             )
        
#         # Removendo colunas 'avg_adt_{}k' adicionadas anteriormente
#         if (idx != 0) and (f'avg_adt_{adt_band}k' 
#                            in pollutant_subsets[poll].columns):
#             pollutant_subsets[poll].drop(columns=[f'avg_adt_{adt_band}k'],
#                                          inplace=True)
    
#     # Remove a primeira coluna 'avg_adt' adicionada
#     pollutant_subsets[poll].drop(columns=['avg_adt'], inplace=True)
        
#     # Preenchendo valores nulos com np.inf (somente colunas 
#     # {micro/meso/bairro/urb}_max podem ter nulos)
#     cols = list(pollutant_subsets[poll].filter(like='k').columns)
#     pollutant_subsets[poll].loc[:,cols] = (
#         pollutant_subsets[poll]
#         .loc[:,cols]
#         .astype(float)
#         .fillna(np.inf)
#     )
        
# del suffix


São adicionados aos subconjuntos de cada poluente os limites de distância correspondentes às diferentes classes de representatividade (micro, meso, bairro e urbano).

A função `add_limites_nas_vias()` percorre as faixas de ADT definidas para cada poluente e associa, a cada estação, os limites de distância referentes ao ADT da via considerada.

Para isso, a função:

1. Define as faixas de ADT consideradas para cada poluente;
2. Verifica se a coluna de ADT correspondente à faixa analisada existe no `GeoDataFrame`;
3. Utiliza a tabela interpolada (`interpolated`) contendo os limites de distância para cada faixa de ADT;
4. Realiza uma junção (`merge`) entre os dados das estações e a tabela de referência, utilizando o valor de ADT como chave;
5. Adiciona ao `subset` as colunas com os limites de distância das classes de representatividade;
6. Remove colunas auxiliares criadas durante o processo;
7. Preenche valores ausentes nos limites de distância com `np.inf`, indicando que não foi definido um limite superior para aquela situação.

O processo é realizado separadamente para cada faixa de ADT e para cada poluente (`CO`, `NO₂`, `O₃`, `PM` e `SO₂`).

Ao final, cada subconjunto contém, além das informações originais e das distâncias até as vias, os limites necessários para verificar posteriormente se cada estação está dentro da faixa de representatividade correspondente à via.

In [27]:
# Iterando sobre os poluentes e seus subconjuntos
def add_limites_nas_vias(subset:gpd.GeoDataFrame,
                         interpolated: pd.DataFrame,
                         poluente:str):
    
    # Dicionário de faixas de ADT (real = valor * 1000)
    pollutants_adt_dict = {'co':[1, 10, 20, 30, 40, 50, 60, np.inf],
                           'so2':[1, 10, 20, 30, 40, 50, 60, np.inf],
                           'no2':[1, 10, 15, 20, 40, 70, 110, np.inf],
                           'pm': [1, 15, 20, 30, 40, 50, 60, 70, 80, np.inf],
                           'o3':[10, 15, 20, 40, 70, 110, np.inf]}
    
    # Iterando sobre os valores de ADT da ref_table para cada poluente
    for idx, adt_band in enumerate(pollutants_adt_dict[poluente][:-1]):
        col_name = f'average_daily_vehicle_count_{adt_band}k'
        
        # Registrando valores de ADT sem vias para cada poluente
        if col_name not in subset.columns:
            print(f"[AVISO] '{col_name}' não existe no subset_{poluente}")
            continue

        # Obtendo a tabela do lado direito (para o merge)
        interp_df = interpolated.copy()
        
        # Renomeando a primeira coluna, para que a função merge não adicione sufixo
        suffix = f"_{adt_band}k"
        interp_df = interp_df.rename(columns={
            col: f"{col}{suffix}" 
            for col
            in interp_df.columns
            if col != 'avg_adt'
        })
        
        # Mesclando colunas com limites de distância para cada ADT e classe representativa
        subset = subset.merge(
            right= interp_df,
            how='left',
            left_on= col_name,
            right_on='avg_adt',
            suffixes=(None, f"_{adt_band}k")
            )
        
        # Removendo colunas 'avg_adt_{}k' adicionadas anteriormente
        if (idx != 0) and (f'avg_adt_{adt_band}k' in subset.columns):
            subset.drop(columns=[f'avg_adt_{adt_band}k'],
                        inplace=True)
    
    # Remove a primeira coluna 'avg_adt' adicionada
    subset.drop(columns=['avg_adt'], inplace=True)
        
    # Preenchendo valores nulos com np.inf (somente colunas 
    # {micro/meso/bairro/urb}_max podem ter nulos)
    cols = list(subset.filter(like='k').columns)
    subset.loc[:,cols] = (
        subset
        .loc[:,cols]
        .astype(float)
        .fillna(np.inf)
    )

    return subset


In [28]:
# Aplicando a função
subset_co = add_limites_nas_vias(subset_co,
                                 interpolated_co,
                                 "co")
subset_no2 = add_limites_nas_vias(subset_no2,
                                 interpolated_no2,
                                 "no2")
subset_o3 = add_limites_nas_vias(subset_o3,
                                 interpolated_o3,
                                 "o3")
subset_pm = add_limites_nas_vias(subset_pm,
                                 interpolated_pm,
                                 "pm")
subset_so2 = add_limites_nas_vias(subset_so2,
                                 interpolated_so2,
                                 "so2")

In [29]:
subset_pm.loc[ subset_pm.ID_OEMA.str.contains('Fercal CRAS', na=False), [col for col in subset_pm.columns if ('min' in col) or ('max' in col)] ]

,meso_min_1k,meso_max_1k,bairro_min_1k,bairro_max_1k,urb_min_1k,urb_max_1k,meso_min_15k,meso_max_15k,bairro_min_15k,bairro_max_15k,...,bairro_min_70k,bairro_max_70k,urb_min_70k,urb_max_70k,meso_min_80k,meso_max_80k,bairro_min_80k,bairro_max_80k,urb_min_80k,urb_max_80k
192,inf,inf,15.00,80.00,80.00,inf,15.00,20.00,15.57,81.14,...,72.73,inf,inf,inf,15.00,130.05,131.29,inf,inf,inf
194,inf,inf,15.00,80.00,80.00,inf,15.00,20.00,15.57,81.14,...,72.73,inf,inf,inf,15.00,130.05,131.29,inf,inf,inf
198,inf,inf,15.00,80.00,80.00,inf,15.00,20.00,15.57,81.14,...,72.73,inf,inf,inf,15.00,130.05,131.29,inf,inf,inf


## 7.4 Verificando se cada via de cada faixa de ADT está dentro dos limites de cada classe

In [30]:
# """Cria uma coluna para cada valor de ADT e poluente, verificando se a via está dentro dos limites dessa classe"""
# # Iterando sobre os poluentes e seus subconjuntos
# for poll, subset in pollutant_subsets.items():
    
#     if poll != 'pm':
        
#         # Iterando sobre os valores de ADT da ref_table para cada poluente
#         for idx, adt_band in enumerate(pollutants_adt_dict[poll][:-1]):
#             distance = f'distance_{adt_band}k'
#             cols = subset.columns
        
#             # MICRO
#             col_micro_min = f'micro_min_{adt_band}k'
#             col_micro_max = f'micro_max_{adt_band}k'
#             if col_micro_min in cols and col_micro_max in cols:
#                 pollutant_subsets[poll][f'rep_micro_{adt_band}k'] = np.where(
#                     (subset[distance] > subset[col_micro_min]) &
#                     (subset[distance] < subset[col_micro_max]),
#                     True,
#                     False)
        
#             # MESO
#             col_meso_min = f'meso_min_{adt_band}k'
#             col_meso_max = f'meso_max_{adt_band}k'
#             if col_meso_min in cols and col_meso_max in cols:
#                 pollutant_subsets[poll][f'rep_meso_{adt_band}k'] = np.where(
#                     (subset[distance] > subset[col_meso_min]) &
#                     (subset[distance] < subset[col_meso_max]),
#                     True,
#                     False)
        
#             # BAIRRO
#             col_bairro_min = f'bairro_min_{adt_band}k'
#             col_bairro_max = f'bairro_max_{adt_band}k'
#             if col_bairro_min in cols and col_bairro_max in cols:
#                 pollutant_subsets[poll][f'rep_bairro_{adt_band}k'] = np.where(
#                     (subset[distance] > subset[col_bairro_min]) &
#                     (subset[distance] < subset[col_bairro_max]),
#                     True,
#                     False)
        
#             # URBANO
#             col_urb_min = f'urb_min_{adt_band}k'
#             col_urb_max = f'urb_max_{adt_band}k'
#             if col_urb_min in cols and col_urb_max in cols:
#                 pollutant_subsets[poll][f'rep_urb_{adt_band}k'] = np.where(
#                     (subset[distance] > subset[col_urb_min]) &
#                     (subset[distance] < subset[col_urb_max]),
#                     True,
#                     False)
            
            
#     else:
#         for idx, adt_band in enumerate(pollutants_adt_dict[poll][:-1]):
#             distance = f'distance_{adt_band}k'
#             cols = subset.columns
        
#             # MICRO
#             col_micro_min = f'micro_min_{adt_band}k'
#             col_micro_max = f'micro_max_{adt_band}k'
#             if col_micro_min in cols and col_micro_max in cols:
#                 pollutant_subsets[poll][f'rep_micro_{adt_band}k'] = np.where(
#                     (subset[distance] > subset[col_micro_min]) &
#                     (subset[distance] < subset[col_micro_max]),
#                     True,
#                     False)
        
#             # MESO
#             col_meso_min = f'meso_min_{adt_band}k'
#             col_meso_max = f'meso_max_{adt_band}k'
#             # Exclui banda 1k do cálculo meso para PM
#             if col_meso_min in cols and col_meso_max in cols and adt_band != 1:
#                 pollutant_subsets[poll][f'rep_meso_{adt_band}k'] = np.where(
#                     (subset[distance] > subset[col_meso_min]) &
#                     (subset[distance] < subset[col_meso_max]),
#                     True,
#                     False)
        
#             # BAIRRO
#             col_bairro_min = f'bairro_min_{adt_band}k'
#             col_bairro_max = f'bairro_max_{adt_band}k'
#             if col_bairro_min in cols and col_bairro_max in cols:
#                 pollutant_subsets[poll][f'rep_bairro_{adt_band}k'] = np.where(
#                     (subset[distance] > subset[col_bairro_min]) &
#                     (subset[distance] < subset[col_bairro_max]),
#                     True,
#                     False)
        
#             # URBANO
#             col_urb_min = f'urb_min_{adt_band}k'
#             col_urb_max = f'urb_max_{adt_band}k'
#             # Exclui banda 80k do cálculo urbano para PM
#             if col_urb_min in cols and col_urb_max in cols and adt_band != 80:
#                 pollutant_subsets[poll][f'rep_urb_{adt_band}k'] = np.where(
#                     (subset[distance] > subset[col_urb_min]) &
#                     (subset[distance] < subset[col_urb_max]),
#                     True,
#                     False)

# # Removendo de variáveis temporárias
# del col_bairro_max, col_bairro_min, col_meso_max, col_meso_min, col_micro_max
# del col_micro_min, col_urb_max, col_urb_min, distance, cols


Nesta etapa, é verificado se a distância entre cada estação de monitoramento e a via mais próxima, para cada faixa de ADT, está dentro dos limites estabelecidos para as diferentes classes de representatividade espacial.

A função `verif_vias_dentro_dos_limites()` utiliza as faixas de ADT definidas para cada poluente e compara a distância calculada para cada via com os limites mínimo e máximo das classes **micro**, **meso**, **bairro** e **urbana**.

Para cada faixa de ADT, são criadas colunas de verificação (`rep_micro`, `rep_meso`, `rep_bairro` e `rep_urb`). Essas colunas recebem:

- `True` quando a distância da estação até a via está dentro dos limites da classe;
- `False` quando a distância está fora dos limites.

As verificações são realizadas separadamente para cada poluente, pois as faixas de ADT consideradas podem ser diferentes entre eles.

Para o material particulado (`PM`), são aplicadas regras específicas: a faixa de ADT de 1k não é utilizada no cálculo da representatividade meso e a faixa de 80k não é utilizada no cálculo da representatividade urbana.

Ao final, a função retorna o GeoDataFrame contendo as novas colunas de representatividade, que serão utilizadas na etapa seguinte para verificar se cada estação possui pelo menos uma via adequada para cada classe de representatividade.

In [31]:
def verif_vias_dentro_dos_limites(subset:gpd.GeoDataFrame,
                                  poluente:str):
    
    # Dicionário de faixas de ADT (value * 1000)
    pollutants_adt_dict = {'co':[1, 10, 20, 30, 40, 50, 60, np.inf],
                           'so2':[1, 10, 20, 30, 40, 50, 60, np.inf],
                           'no2':[1, 10, 15, 20, 40, 70, 110, np.inf],
                           'pm': [1, 15, 20, 30, 40, 50, 60, 70, 80, np.inf],
                           'o3':[10, 15, 20, 40, 70, 110, np.inf]}
    
    if poluente != 'pm':
        
        # Iterando sobre os valores de ADT da ref_table para cada poluente
        for idx, adt_band in enumerate(pollutants_adt_dict[poluente][:-1]):
            distance = f'distance_{adt_band}k'
            cols = subset.columns
        
            # MICRO
            col_micro_min = f'micro_min_{adt_band}k'
            col_micro_max = f'micro_max_{adt_band}k'
            if col_micro_min in cols and col_micro_max in cols:
                subset[f'rep_micro_{adt_band}k'] = np.where(
                    (subset[distance] > subset[col_micro_min]) &
                    (subset[distance] < subset[col_micro_max]),
                    True,
                    False)
        
            # MESO
            col_meso_min = f'meso_min_{adt_band}k'
            col_meso_max = f'meso_max_{adt_band}k'
            if col_meso_min in cols and col_meso_max in cols:
                subset[f'rep_meso_{adt_band}k'] = np.where(
                    (subset[distance] > subset[col_meso_min]) &
                    (subset[distance] < subset[col_meso_max]),
                    True,
                    False)
        
            # BAIRRO
            col_bairro_min = f'bairro_min_{adt_band}k'
            col_bairro_max = f'bairro_max_{adt_band}k'
            if col_bairro_min in cols and col_bairro_max in cols:
                subset[f'rep_bairro_{adt_band}k'] = np.where(
                    (subset[distance] > subset[col_bairro_min]) &
                    (subset[distance] < subset[col_bairro_max]),
                    True,
                    False)
        
            # URBANO
            col_urb_min = f'urb_min_{adt_band}k'
            col_urb_max = f'urb_max_{adt_band}k'
            if col_urb_min in cols and col_urb_max in cols:
                subset[f'rep_urb_{adt_band}k'] = np.where(
                    (subset[distance] > subset[col_urb_min]) &
                    (subset[distance] < subset[col_urb_max]),
                    True,
                    False)
            
            
    else:
        for idx, adt_band in enumerate(pollutants_adt_dict[poluente][:-1]):
            distance = f'distance_{adt_band}k'
            cols = subset.columns
        
            # MICRO
            col_micro_min = f'micro_min_{adt_band}k'
            col_micro_max = f'micro_max_{adt_band}k'
            if col_micro_min in cols and col_micro_max in cols:
                subset[f'rep_micro_{adt_band}k'] = np.where(
                    (subset[distance] > subset[col_micro_min]) &
                    (subset[distance] < subset[col_micro_max]),
                    True,
                    False)
        
            # MESO
            col_meso_min = f'meso_min_{adt_band}k'
            col_meso_max = f'meso_max_{adt_band}k'
            # Exclui banda 1k do cálculo meso para PM
            if col_meso_min in cols and col_meso_max in cols and adt_band != 1:
                subset[f'rep_meso_{adt_band}k'] = np.where(
                    (subset[distance] > subset[col_meso_min]) &
                    (subset[distance] < subset[col_meso_max]),
                    True,
                    False)
        
            # BAIRRO
            col_bairro_min = f'bairro_min_{adt_band}k'
            col_bairro_max = f'bairro_max_{adt_band}k'
            if col_bairro_min in cols and col_bairro_max in cols:
                subset[f'rep_bairro_{adt_band}k'] = np.where(
                    (subset[distance] > subset[col_bairro_min]) &
                    (subset[distance] < subset[col_bairro_max]),
                    True,
                    False)
        
            # URBANO
            col_urb_min = f'urb_min_{adt_band}k'
            col_urb_max = f'urb_max_{adt_band}k'
            # Exclui banda 80k do cálculo urbano para PM
            if col_urb_min in cols and col_urb_max in cols and adt_band != 80:
                subset[f'rep_urb_{adt_band}k'] = np.where(
                    (subset[distance] > subset[col_urb_min]) &
                    (subset[distance] < subset[col_urb_max]),
                    True,
                    False)
    
    return subset

# Aplicando função
subset_co = verif_vias_dentro_dos_limites(subset_co,
                                          "co")
subset_no2 = verif_vias_dentro_dos_limites(subset_no2,
                                           "no2")
subset_o3 = verif_vias_dentro_dos_limites(subset_o3,
                                          "o3")
subset_pm = verif_vias_dentro_dos_limites(subset_pm,
                                          "pm")
subset_so2 = verif_vias_dentro_dos_limites(subset_so2,
                                           "so2")

In [32]:
subset_pm.loc[
    subset_pm['ID_OEMA'].str.contains('Fercal Escola', na=False)
]

,UF,CIDADE,CD_MUN,ID_OEMA,ID_MMA,ID_MMA_COMPLETO,PROPRIETARIO,PROP_ENTIDADE,OPERADOR,OP_ENTIDADE,...,rep_bairro_50k,rep_urb_50k,rep_meso_60k,rep_bairro_60k,rep_urb_60k,rep_meso_70k,rep_bairro_70k,rep_urb_70k,rep_meso_80k,rep_bairro_80k
193,DF,Fercal,Nao declarado,Fercal Escola,DF0002,DF0002RA001,IBRAM,Publica,CIPLAN,Privada,...,False,True,False,False,True,False,True,False,False,True
195,DF,Fercal,Nao declarado,Fercal Escola,DF0002,DF0002RA002,IBRAM,Publica,CIPLAN,Privada,...,False,True,False,False,True,False,True,False,False,True


In [33]:
subset_pm.loc[subset_pm.ID_OEMA.str.contains('Fercal Escola')].iloc[[1]][[col for col in list(subset_pm.columns) if (('meso' in col)&(('min' in col)|('max'in col))) ]]

,meso_min_1k,meso_max_1k,meso_min_15k,meso_max_15k,meso_min_20k,meso_max_20k,meso_min_30k,meso_max_30k,meso_min_40k,meso_max_40k,meso_min_50k,meso_max_50k,meso_min_60k,meso_max_60k,meso_min_70k,meso_max_70k,meso_min_80k,meso_max_80k
195,inf,inf,15.00,20.00,15.00,22.29,15.00,37.31,15.00,46.87,15.00,54.28,15.00,64.33,15.00,72.73,15.00,130.05


## 7.5 Verificando se cada estação possui ao menos uma via de ao menos uma faixa de ADT que se encontre dentro dos limites de cada classe de representatividade

Para a verificação, são utilizadas as colunas `rep_micro`, `rep_meso` e `rep_urb`, geradas na etapa anterior. Essas colunas indicam, para cada faixa de ADT, se a distância entre a estação e a via correspondente está dentro dos limites definidos para as classes **micro**, **meso** e **urbana**.

A função `safe_any()` é utilizada como uma etapa auxiliar. Ela verifica, para cada estação, se existe **pelo menos uma coluna correspondente à classe analisada contendo o valor `True`**. Caso não existam colunas correspondentes, a função retorna `False`, evitando que a verificação produza um resultado incorreto.

Dessa forma, esta etapa permite identificar quais estações possuem pelo menos uma via que atende aos critérios de distância para cada classe de representatividade, servindo como uma verificação final da disponibilidade de vias adequadas para a classificação das estações.

In [34]:
# Atribuindo GeoDataFrames do dicionário para variáveis individuais
# subset_co = pollutant_subsets['co']
# subset_no2 = pollutant_subsets['no2']
# subset_o3 = pollutant_subsets['o3']
# subset_pm = pollutant_subsets['pm']
# subset_so2 = pollutant_subsets['so2']

# Função auxiliar para verificar se existem colunas de cada escala
def safe_any(df, like_str):
    """
    Filtra todas as colunas de um dataframe com uma substring comum no nome
    e verifica se há pelo menos um valor True em cada linha, retornando uma série de True.
    Caso contrário ou se estiver vazio, retorna uma série de False.
    
    Essa função é necessária porque, se não houver colunas no dataframe filtrado,
    a função .all retorna uma série de True, o que é enganoso.
    
    Parâmetros
    ----------
    df : dataframe 
        DataFrame com várias colunas contendo um elemento repetido no nome.
    like_str: str
        Substring comum nos nomes das colunas do DataFrame.

    Retorna
    -------
    filtered : série de booleans
        Série booleana com o mesmo tamanho que o DataFrame original.
    """
    # Filtra todas as colunas que contêm a string específica no nome
    filtered = df.filter(like=like_str)
    
    # Se não houver colunas correspondentes, retorna uma série de False
    if filtered.shape[1] == 0:
        return pd.Series([False] * len(df), index=df.index)
        
    return filtered.any(axis='columns')


# Aplicando a função para verificar a validade de cada rep_{classe} --------------

# CO ---------------------------------
subset_co['rep_micro_any'] = safe_any(subset_co, 'rep_micro')
subset_co['rep_bairro_any'] = safe_any(subset_co, 'rep_bairro')

# NO2 -------------------------------
subset_no2['rep_micro_any'] = safe_any(subset_no2, 'rep_micro')
subset_no2['rep_bairro_any'] = safe_any(subset_no2, 'rep_bairro')
subset_no2['rep_urb_any'] = safe_any(subset_no2, 'rep_urb')

# O3 --------------------------------
subset_o3['rep_bairro_any'] = safe_any(subset_o3, 'rep_bairro')
subset_o3['rep_urb_any'] = safe_any(subset_o3, 'rep_urb')

# PM -------------------------------
subset_pm['rep_meso_any'] = safe_any(subset_pm, 'rep_meso')
subset_pm['rep_bairro_any'] = safe_any(subset_pm, 'rep_bairro')
subset_pm['rep_urb_any'] = safe_any(subset_pm, 'rep_urb')

# SO2 -------------------------------------------
subset_so2['rep_micro_any'] = safe_any(subset_so2, 'rep_micro')
subset_so2['rep_bairro_any'] = safe_any(subset_so2, 'rep_bairro')


In [35]:
subset_pm.loc[subset_pm.ID_OEMA.str.contains('Fercal Escola')].iloc[[1]][[col for col in list(subset_pm.columns) if 'any' in col]]

,rep_meso_any,rep_bairro_any,rep_urb_any
195,False,True,True


## 7.6 Classificando a representatividade espacial de todas as estações

A função `def classify_spatial_rep` verifica valores **True** nas colunas referentes a cada escala espacial e atribui a cada estação a escala mais restritiva, seguindo a ordem abaixo:    
                    
                    1º  >   2º   >    3º    >   4º
                    
                    MICRO > MESO > BAIRRO > URBANA
    
A lógica é que, se houver pelo menos uma via que determine que a estação é representativa espacialmente para uma classe mais restritiva, então ela é considerada representativa nessa escala.


In [36]:
# %% CLASSIFICANDO O STATUS DE REPRESENTATIVIDADE DE CADA ESTAÇÃO ----------------------------
def classify_spatial_rep(subset):
    """
    Parâmetros
    ----------
    subset : geodataframe 
        Subconjunto das estações de monitoramento, referente a um único poluente

    Retorna
    -------
    subset : geodataframe
        Subconjunto de entrada com uma nova coluna chamada 'REP_ESPACIAL_NAME'
    """
    # conditions = []
    # choices = []
    
    # if 'rep_micro_any' in subset.columns:
    #     conditions.append(subset['rep_micro_any'] == True)
    #     choices.append('micro')
        
    # if 'rep_meso_any' in subset.columns:
    #     conditions.append(subset['rep_meso_any'] == True)
    #     choices.append('meso')
    
    # if 'rep_bairro_any' in subset.columns:
    #     conditions.append(subset['rep_bairro_any'] == True)
    #     choices.append('bairro')
        
    # if 'rep_urb_any' in subset.columns:
    #     conditions.append(subset['rep_urb_any'] == True)
    #     choices.append('urbana')

    # # Atribui o nome da escala mais restritiva encontrada, ou 'não representativo' caso nenhuma se aplique
    # subset['REP_ESPACIAL_NAME'] = np.select(
    #     conditions,
    #     choices,
    #     default='não representativo'
    # )
    
    # return subset

    """============ ALTERAÇÃO IGOR =============="""

    # Assegurando de não estar sobrescrevendo o subset original
    subset = subset.copy()
    
    # Criando lista de possíveis colunas a considerar
    # A lista foi criada do menos restritivo para mais restritivo
    rep_possibilities = {
        'rep_urb_any': 'urbana',
        'rep_bairro_any': 'bairro',
        'rep_meso_any': 'meso',
        'rep_micro_any': 'micro',
    }

    # Filtrando as que estão no subset
    rep_possibilities = {
        col: item
        for col, item in rep_possibilities.items()
        if col in subset.columns
    }

    # Definindo variável de resultado como lista vazia do tamanho de subset
    rep_spatial_name_serie = pd.Series(index=range(len(subset)), dtype='str')
    rep_spatial_name_serie.loc[:] = 'não representativo'

    # Adicionando resultados do menos restritivo pro mais restritivo
    for header, item in rep_possibilities.items():
        # Adicionando só onde os valores da coluna forem iguais a True
        rep_spatial_name_serie.loc[subset[header]] = item

    # Adicionando no subset final
    subset.loc[:, 'REP_ESPACIAL_NAME'] = rep_spatial_name_serie

    return subset

    """============ ALTERAÇÃO IGOR =============="""
        

# Aplicando a classificação de representatividade espacial para cada poluente
subset_co = classify_spatial_rep(subset_co)
subset_no2 = classify_spatial_rep(subset_no2)
subset_o3 = classify_spatial_rep(subset_o3)
subset_pm = classify_spatial_rep(subset_pm)
subset_so2 = classify_spatial_rep(subset_so2)


In [37]:
subset_pm.loc[subset_pm.ID_OEMA.str.contains('Fercal Escola')].iloc[[1]][[col for col in list(subset_pm.columns) if 'REP_ESPACIAL_NAME' in col]]

,REP_ESPACIAL_NAME
195,bairro


## 7.7. Criando buffers de representatividade espacial para cada estação

A função `def create_buffered_gdf` cria colunas com tamanhos de buffer para um GeoDataFrame, agrupando por códigos EPSG, com a opção de criar geometrias de buffer.

In [38]:
#%% BUFFERS DE REPRESENTATIVIDADE -----------------------------------------------------
"""microscale: < 100 m
   mesoscale: 100 m < x < 500 m
   escala de bairro: 500 m < x < 4000 m
   escala urbana: 4000 m < x < 50000 m
""" 

def create_buffered_gdf(subset,
                        buffer_sizes,
                        target_crs="EPSG:4326",
                        create_buffer=True):
    """

    Parâmetros:
        subset (GeoDataFrame): GeoDataFrame de entrada com as colunas 'EPSG', 
        'REP_ESPACIAL_NAME' e 'geometry'.
        
        buffer_sizes (dict): Dicionário que mapeia os valores de `REP_ESPACIAL_NAME`
        para tamanhos de buffer (em metros).
        
        target_crs (str): CRS (sistema de referência espacial) para reprojetar o 
        GeoDataFrame final. Padrão é 'EPSG:4326'.
        
        create_buffer (bool): Verifica se as geometrias de buffer devem ser criadas. 
        Padrão é True.

    Retorna:
        GeoDataFrame: O GeoDataFrame de entrada com colunas adicionais:
            "REP_ESPACIAL": int.
                Tamanho do buffer de representatividade espacial
            "REP_ESPACIAL_NAME": str.
                Nome da classe de representatividade espacial 
            "REP_ESPACIAL_BUFFER": polígono. 
                Geometrias de buffer ao redor das estações no CRS de destino.
    """
    
    buffered_gdfs = []

    for epsg in subset['EPSG'].unique():
        # Filtra o subset e projeta para o respectivo EPSG
        gdf_epsg = subset[subset['EPSG'] == epsg].to_crs(epsg)

        # Remove a coluna REP_ESPACIAL existente
        gdf_epsg = gdf_epsg.drop(columns=['REP_ESPACIAL'])
        
        # Mapeia os tamanhos dos buffers com base no nome da representatividade
        gdf_epsg['REP_ESPACIAL'] = (
            gdf_epsg['REP_ESPACIAL_NAME']
            .map(buffer_sizes)
            .fillna(0)
        )
        
        # Cria as geometrias de buffer se create_buffer for True
        if create_buffer == True:
            gdf_epsg['REP_ESPACIAL_BUFFER'] = (
                gdf_epsg
                .geometry
                .buffer(gdf_epsg['REP_ESPACIAL'])
                .to_crs(target_crs)
            )

        # Reprojeta para o CRS de destino
        gdf_epsg = gdf_epsg.to_crs(target_crs)

        # Adiciona à lista
        buffered_gdfs.append(gdf_epsg)

    # Combina todos os GeoDataFrames com buffer
    buffered_subset = gpd.GeoDataFrame(
        pd.concat(buffered_gdfs, ignore_index=True),
        crs=target_crs
    )

    return buffered_subset


# Dicionário de tamanhos de buffer de acordo com REP_ESPACIAL_NAME
buffer_sizes = {
    'urbana': 50000,
    'bairro': 4000,
    'meso': 500,
    'micro': 100,
}

# Aplicando a função
# CO 
buffered_subset_co = create_buffered_gdf(subset_co,
                                         buffer_sizes,
                                         target_crs='EPSG:4326',
                                         create_buffer=False)
# NO2 
buffered_subset_no2 = create_buffered_gdf(subset_no2,
                                          buffer_sizes,
                                          target_crs='EPSG:4326',
                                          create_buffer=False)
# O3
buffered_subset_o3 = create_buffered_gdf(subset_o3,
                                         buffer_sizes,
                                         target_crs='EPSG:4326',
                                         create_buffer=False)
# PM 
buffered_subset_pm = create_buffered_gdf(subset_pm,
                                         buffer_sizes,
                                         target_crs='EPSG:4326',
                                         create_buffer=False)
# SO2 
buffered_subset_so2 = create_buffered_gdf(subset_so2,
                                          buffer_sizes,
                                          target_crs='EPSG:4326',
                                          create_buffer=False)


# 8.0 Formatação dos outputs

Nesta etapa, são organizados e preparados os resultados finais para facilitar sua utilização e interpretação.

Primeiramente, os resultados referentes aos diferentes poluentes (`CO`, `NO₂`, `O₃`, `PM` e `SO₂`) são unidos em um único GeoDataFrame, formando o conjunto final de estações analisadas.

Em seguida, são removidas as colunas auxiliares utilizadas durante os cálculos das etapas anteriores. Essas colunas incluem informações intermediárias relacionadas às representatividades (`rep`), aos limites mínimo e máximo (`min` e `max`), às faixas de ADT (`k`) e outras informações utilizadas no cálculo da distância até indústrias.

O objetivo dessa etapa é manter no resultado final apenas as informações necessárias para a apresentação e utilização dos dados, evitando que o output contenha colunas intermediárias utilizadas apenas durante o processamento.

In [39]:
# 1) estacoes_completa
# Unindo poluentes em um gdf final completo
buffered_stations = pd.concat([buffered_subset_co,
                               buffered_subset_no2,
                               buffered_subset_o3,
                               buffered_subset_pm,
                               buffered_subset_so2])


# 2) rep_espacial

# Removendo colunas auxiliares
def drop_aux_cols(subset):
    return subset.drop(columns= (list(subset
                                      .filter(like='rep')
                                      .columns) +
                                 list(subset
                                      .filter(like='min')
                                      .columns) +
                                 list(subset
                                      .filter(like='max')
                                      .columns) +
                                 list(subset
                                      .filter(like='k')
                                      .columns) +
                                 ['EPSG','distance_to_industry',
                                  'Razão Social', 'industry_geom']
                                 )
                       )
# Aplicando função
filtered_stations = drop_aux_cols(buffered_stations) 



# 9.0 Verificando outputs

## 9.1 Estações e representatividade espacial

In [40]:
filtered_stations['REP_ESPACIAL_NAME'].unique()

array(['bairro', 'micro', 'meso'], dtype=object)

In [41]:
filtered_stations.head()

,UF,CIDADE,CD_MUN,ID_OEMA,ID_MMA,ID_MMA_COMPLETO,PROPRIETARIO,PROP_ENTIDADE,OPERADOR,OP_ENTIDADE,...,REALOCACAO,OBS_CALIBRACAO,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,REP_ESPACIAL_DECLARADA,OPERACAO,geometry,REP_ESPACIAL_NAME,REP_ESPACIAL
0,BA,Dias d'Ávila,2910057,LEANDRINHO,BA0008,BA0008ND007,Nao declarado,Nao declarado,Nao declarado,Nao declarado,...,Nao declarado,Nao declarado,Sim,NaN,Nao declarado,Nao declarado,Nao declarado,POINT (-38.37487 -12.64159),bairro,4000
1,BA,Dias d'Ávila,2910057,CONCORDIA,BA0007,BA0007ND007,Nao declarado,Nao declarado,Nao declarado,Nao declarado,...,Nao declarado,Nao declarado,Sim,NaN,Nao declarado,Nao declarado,Nao declarado,POINT (-38.32561 -12.60405),bairro,4000
2,BA,Dias d'Ávila,2910057,COBRE,BA0002,BA0002ND007,Nao declarado,Nao declarado,Nao declarado,Nao declarado,...,Nao declarado,Nao declarado,Sim,NaN,Nao declarado,Nao declarado,Nao declarado,POINT (-38.35853 -12.63026),bairro,4000
3,BA,Dias d'Ávila,2910057,FUTURAMAI,BA0009,BA0009ND007,Nao declarado,Nao declarado,Nao declarado,Nao declarado,...,Nao declarado,Nao declarado,Sim,NaN,Nao declarado,Nao declarado,Nao declarado,POINT (-38.37157 -12.66899),bairro,4000
4,BA,São Sebastião do Passé,2929503,LAMARAO,BA0004,BA0004ND007,Nao declarado,Nao declarado,Nao declarado,Nao declarado,...,Nao declarado,Nao declarado,Sim,NaN,Nao declarado,Nao declarado,Nao declarado,POINT (-38.39856 -12.59548),bairro,4000


## 9.2 Planilha completa com colunas auxiliares

In [42]:
buffered_stations.shape

(1684, 204)

In [43]:
buffered_stations.loc[buffered_stations.ID_OEMA.str.contains('Fercal Escola')].iloc[[1]][[col for col in list(buffered_stations.columns) if 'distance' in col]]

,distance_1k,distance_10k,distance_20k,distance_30k,distance_40k,distance_50k,distance_60k,distance_to_industry,distance_15k,distance_70k,distance_110k,distance_80k
195,1966.90,NaN,2231.48,2850.42,8868.96,14556.44,10564.59,9245.52,3266.66,6890.36,NaN,4667.33


# 10.0 Salvando outputs

### --> Verificar se existe outputs_path para salvar os resultados
### --> Perguntar se o usuário verá o resultado dessa forma mesmo

In [44]:
# Salvando o GeoDataFrame com a indústria mais próxima de cada estação
buffered_stations.to_parquet(outputs_path + '/estacoes_completa.parquet')

# Salvando o GeoDataFrame de input com as estações, sua classificação de 
# representatividade espacial e o tamanho do buffer
"""Todas as colunas da planilha de estações + ['REP_ESPACIAL','REP_ESPACIAL_NAME']"""
filtered_stations.to_csv(outputs_path + '/rep_espacial.csv')


In [45]:
# roads.to_file(outputs_path + '/roads_adt_test.gpkg')
# buffered_stations.drop(columns=['industry_geom']).to_file(outputs_path + '/estacoes_completa.gpkg')